# NVCL-KIT Example: Generate summary data

Simple examples of generating summary data similar to what you would see in the TSG Summary screen.

In [ ]:
# Import the necessary modules
import pandas as pd
from nvcl_kit.generators import gen_summary_dataframe
from nvcl_kit.param_builder import param_builder
from nvcl_kit.reader import NVCLReader

# Initialise NVCL reader
param = param_builder("NSW")
if not param:
    print("Error: failed to setup connection parameters")

reader = NVCLReader(param)
if not reader.wfs:
    print("Error: Cannot contact service")

## Start depth options

The start_depth values can be configured to start at a whole number (`'floor'`), rounded to 2 places (`'round'`), or to start from the first depth (`'min'` or `None`).

The following will generate summary data with depths starting from the minimum depth value of the dataset.

In [ ]:
boreholeid = "MIN_060160"
bin_size = 1.0
min_item_pct = 0.05
scalar_set = "ujCLST"

# Starting at the minimum depth value (start_depth="min")
meta, df = next(gen_summary_dataframe(reader=reader, nvcl_id_list=[boreholeid], scalar_set=scalar_set, scalar_level="group", start_depth="min", weighted=True, resolution=bin_size, min_item_pct=min_item_pct, continue_on_missing=True), None)
if isinstance(df, pd.DataFrame):
    # Display subset of columns for the first 10 rows of the dataframe
    print(df[["BoreholeID", "StartDepth", "EndDepth", "PLAGIOCLASE","SILICA", "WHITE-MICA"]].head(10))
else:
    print(f"Failed to generate summary of {boreholeid}")

Generate summary data with depths starting at the minimum depth, rounded to 2 decimal places.

In [ ]:
# Starting at the minimum depth value rounded to 2 decimal places (start_depth="round")
meta, df = next(gen_summary_dataframe(reader=reader, nvcl_id_list=[boreholeid], scalar_set=scalar_set, scalar_level="group", start_depth="round", weighted=True, resolution=bin_size, min_item_pct=min_item_pct, continue_on_missing=True), None)
if isinstance(df, pd.DataFrame):
    print(df[["BoreholeID", "StartDepth", "EndDepth", "PLAGIOCLASE","SILICA", "WHITE-MICA"]].head(10))
else:
    print(f"Failed to generate summary of {boreholeid}")

Generate summary starting at the floor of the minimum depth value. e.g. if data starts at `106.21195` the first bin will start at `106`.

In [ ]:
# Starting at the floor of the minimum depth value (start_depth="floor")
meta, df = next(gen_summary_dataframe(reader=reader, nvcl_id_list=[boreholeid], scalar_set=scalar_set, scalar_level="group", start_depth="floor", weighted=True, resolution=bin_size, min_item_pct=min_item_pct, continue_on_missing=True), None)
if isinstance(df, pd.DataFrame):
    print(df[["BoreholeID", "StartDepth", "EndDepth", "PLAGIOCLASE","SILICA", "WHITE-MICA"]].head(10))
else:
    print(f"Failed to generate summary of {boreholeid}")

## Summary Plot Example

The following will create a summary plot similar to what you will find in The Spectral Geologist™ software.

In [ ]:
from matplotlib.ticker import MultipleLocator

# Generate the summary data and then plot it 
meta, df = next(gen_summary_dataframe(reader=reader, nvcl_id_list=[boreholeid], scalar_set=scalar_set, scalar_level="group", start_depth="floor", weighted=True, resolution=bin_size, min_item_pct=min_item_pct, continue_on_missing=True), None)

# Create a colour map using the colour values from the metadata classifications
colors = {c[0]: c[1]["colour"] for c in meta["classifications"].items()}

# Create the summary plot
ax = df.plot.bar(x="StartDepth", y=df.columns[3:], stacked=True, figsize=(12,4), width=1.0, color=colors, ylim=(0,100))
ax.legend(title="HyLogger™ Spectral Groups", loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=4, frameon=False)
ax.set_title(f"{boreholeid}: Spatial Summary (Bin={bin_size}, MinBin={min_item_pct}, {scalar_set} {meta['scalar_algorithm_version']}, Mineral Group)")
ax.set_ylabel("Bin Spectral Contribution")
ax.set_xlabel("Depth (m)")
ax.axes.xaxis.set_major_locator(MultipleLocator(10))
ax.axes.xaxis.set_minor_locator(MultipleLocator(2))